# MMML Lab Assigment 6: Linear Algebra Methods/Applications

The aim of this lab assigment is to study and implement various linear algebra methods and applications, including Spectral Clustering, PCA/LDA and Procrustes problem. 

We will discuss:
* Spectral Clustering on example of the hard-to-separate data
* PCA and LDA algorithms on the famous faces dataset
* ICP and Procrustes Problem for Point Cloud Registration task

Answering all questions will bring 6 points out of the total course grade.

### Instructions 

1.   Form a team of two
2.   Rename the ipynb file to `Name1_Name2_MMML_Lab6.ipynb`
3.   Indicate team members at the top
4.   Provide your solutions (code or explanation) as necessary; do not reshuffle the cell order! You are welcome to optimize the existing commands if you want 
5.   Please execute all the cells before submission; make sure there are no errors, all plots have been generated, and all numerical answers calculated. Also, do not make *looong* prints
6.   Submit your notebook to **cms** along with the pdf output.  


### Import needed libraries

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import trimesh
import scipy
import sklearn
import sklearn.datasets
import sklearn.cluster
from sklearn.svm import SVC
from scipy.spatial.transform import Rotation
from typing import Callable, Tuple, List

## 1. Spectral Clustering (2 pts)

**credits to Yurii Antentyk for previous version of this assigment!**

Clustering is a collective term for algorithms that partition the input dataset into some groups (called _clusters_) based on the similarities of the data points. You are already familiar with some of the clustering algorithms (such as k-means clustering). Note that clustering algorithms are part of **unsupervised** learning algorithms (i.e. there are no labels associated with each data point).

The main goal of this assignment is to understand what spectral clustering is and, most importantly, **why** it works. You will implement its basic steps, test your implementation on a model dataset, and compare the results depending on different hyperparameters.

### Spectral Clustering References

Before talking about spectral clustering, here are the references that could help you to understand it better:

* Kevin P. Murphy. Probabilistic Machine Learning: An Introduction, MIT, 2022. Section 21.5 https://probml.github.io/pml-book/book1.html
* Ulrike von Luxburg. A Tutorial on Spectral Clustering. https://arxiv.org/pdf/0711.0189.pdf

To understand the challenges the clustering algorithms need to overcome, let's consider the following toy dataset:

In [ ]:
X, Y = sklearn.datasets.make_moons(n_samples=500, noise=0.05, random_state=42)

And let's visualize it:

In [ ]:
def plot_clustering(X: np.ndarray, labels1: np.ndarray, text1: str, labels2: np.ndarray, text2: str):
    fig, [ax1, ax2] = plt.subplots(figsize=(10, 5), nrows=1, ncols=2)
    colors = np.array(['blue', 'orange'])

    ax1.set_title(text1)
    ax1.scatter(X[:,0], X[:,1], c=colors[labels1])

    ax2.set_title(text2)
    ax2.scatter(X[:,0], X[:,1], c=colors[labels2])

    plt.show()


plot_clustering(X, Y, 'Dataset as a mixture of 2 distributions', np.zeros_like(Y), 'Input to clustering algorithm')

As you can see, the dataset is a mixture of some 2 non-linear distributions. We would naturally want them to be clustered as on the left picture. However, we don't know the parameters of the distribution and receive the dataset as a collections of points (right picture)

### 1.1 K-Means (0.25 pts)

Why Bother With Spectral Clustering? Indeed, we know k-means, why not to use it?

* Train K-Means classifier with 2 clusters. Visualize and comment on its results.

**Note:** no need to write K-Means by hand, clustering can be done with 1 line of code :)

**P.S.** you might want to use ```sklearn.cluster.KMeans```

**P.S.** [here](https://scikit-learn.org/stable/auto_examples/cluster/plot_cluster_comparison.html) is a good comparison of various clustering algorithms on different datasets. Take a look at ```MiniBatch KMeans``` (which is a slight modification of kmeans).

---


In [ ]:
# perform clustering via k-means

# ========= YOUR CODE STARTS HERE ========= #

# ========== YOUR CODE ENDS HERE ========== #

In [ ]:
# check your clustering via k-means

# ========= YOUR CODE STARTS HERE ========= #

# ========== YOUR CODE ENDS HERE ========== #

* Did it perform well? What are the necessary conditions of input data for kmeans clustering to show good results? (2 sentences max)

### Similarity and Weights Matrix

Spectral clustering closely works with the notion of *similarity* $s_{ij} \geq 0$ between data points (with close points having high similarity and distant points having low similarity).

Based on similarities, we form a **weight matrix** $W$, which is then passed to the spectral clustering algorithm. One important note is that $W$ needs to model the local neighborhood relationships between the data points (it also need to satisfy $W_{ij} \geq 0$ and $W_{ij} = W_{ji}$ for every $i,j$).

There are a few ways to construct a **weight matrix**:

1. **$\epsilon$-neighbourhood**. We say that

$$
\begin{equation}
  \quad \quad W_{ij}=\begin{cases}
    1, & \text{if $\lVert X_i - X_j \rVert < \epsilon$},\\
    0, & \text{otherwise}.
  \end{cases}
\end{equation}
$$

2. **$k$-nearest neighbor**. We say that $W_{ij} = 1$ if $i$ is among $k$ nearest neighbors of $j$ or vice versa (to make this relation symmetric), and $0$ otherwise

3. **Gaussian similarity**. We say that $W_{ij} = e^{- \gamma \lVert X_i - X_j \rVert ^2}$ (with $\gamma \geq 1$)


**P.S.** In case you need a more thorough explanation, please read ```Section 2``` of [A Tutorial on Spectral Clustering](https://arxiv.org/pdf/0711.0189.pdf), which covers this in much greater detail

### Visualizing the Epsilon Neighbourhood

Here is an example of $\epsilon$-neighborhood for $\epsilon = 0.15$:

In [ ]:
eps = 0.15

colors = np.array(['blue', 'orange'])
plt.figure()
plt.title('Visualizing epsilon neighborhood graph for eps=%s' % (eps))

for i in range(X.shape[0]):
    for j in range(i + 1, X.shape[0]):
        d = math.sqrt(np.sum((X[i] - X[j]) ** 2))
        if d <= eps:
            plt.plot((X[i,0], X[j,0]), (X[i,1], X[j,1]), zorder=-1, c='black', alpha=0.5)

plt.scatter(X[:,0], X[:,1], c=colors[Y], zorder=1)
plt.show()

### Understanding Gaussian Similarity

As you can see from the previous picture, the **$\epsilon$-neighbourhood** weight matrix is actually the graph, which models the neighbourhood relation between the data points (same is true for **$k$-nearest neighbor** weight matrix).

However, with **Gaussian similarity** the graph becomes fully connected, i.e. all edges have some positive weight. If you think about the formula, you'll see that close points will get values close to $1$, while distant points will get the values close to $0$ (with exponential decay). Note that with increasing gamma the function becomes even more steep:

In [ ]:
fig, axis = plt.subplots(figsize=(15, 5), nrows=3, ncols=1, sharex=True, sharey=True)
gammas = [1, 5, 10]

assert len(axis) == len(gammas)

x = np.linspace(0, 10, 100)

for ax, gamma in zip(axis, gammas):
    y = np.exp(-gamma * x)
    ax.plot(x, y, label='gamma=%s' % (gamma, ))
    ax.legend()

fig.suptitle('Comparing Gaussian Similarity (y) as a function of distance (x) for different gammas')
axis[-1].set_xlabel('Distance')

plt.show()

### 1.2 Calculating Gaussian Similarity (0.25 pts)

* write a function for calculating **Gaussian Similarity** weights matrix for the given input data

**Note:** in ```sklearn``` documentation on spectral clustering:
* **Gaussian Similarity** $\approx$ **rbf kernel**
* **Weights matrix** is identical to **affinity matrix**

In [ ]:
def rbf_affinity_matrix(X: np.array, gamma: float) -> np.array:
    # ========= YOUR CODE STARTS HERE ========= #
    
    # ========== YOUR CODE ENDS HERE ========== #
    return weights

In [ ]:
# test code
for gamma in range(1, 15, 2):
    expected_affinity = sklearn.cluster.SpectralClustering(n_clusters=2, affinity='rbf', gamma=gamma).fit(X).affinity_matrix_
    actual_affinity = rbf_affinity_matrix(X, gamma)

    assert expected_affinity.shape == actual_affinity.shape, 'Wrong affinity matrix shape, expected %s, got %s (gamma=%s)' % (expected_affinity.shape, actual_affinity.shape, gamma)
    assert np.sum(np.abs(expected_affinity - actual_affinity) > 1e-6) == 0, 'Some RBF affinity matrix values are wrong (gamma=%s)' % (gamma, )

### Laplacian Matrix

Actually, spectral clustering works not with affinity matrix, but with the **Laplacian** matrix, which is defined as follows:

$D$ &mdash; degree matrix (diagonal), $D_{ii} = \sum\limits_j A_{ij}$ (where $A$ is the affinity matrix)

$L$ &mdash; unnormalized Laplacian, $L = D - A$

$L_{\mathrm{sym}}$ &mdash; normalized Laplacian, $L_{\mathrm{sym}} = D^{-1/2} L D^{-1/2}$, with $D^{-1/2}$ being a diagonal matrix with diagonal entries $\frac{1}{\sqrt{D_{ii}}}$

### Properties of the Laplacian matrix

**Note:** this is very closely related to ```Section 3.1``` of [A Tutorial on Spectral Clustering](https://arxiv.org/pdf/0711.0189.pdf)

The Laplacian $L$ (and $L_{\mathrm{sym}}$) has some nice properties:

1. For every vector $f \in \mathbb{R}^n$:

    $f^TLf = \frac{1}{2}\sum\limits_{i,j=1}^{n} w_{ij}(f_i - f_j)^2$
2. Because $w_{ij} \geq 0$ and $w_{ij} = w_{ji}$, $L$ is a symmetric positive semi-definite (previous equation is literally the definition), with $f \equiv const$ being an element of the null-space
3. From 2. we conclude that the eigenvalues of $L$ are $\geq 0$
4. It can be argued that the multiplicity of the eigenvalue $0$ of $L$ is equal to the number of connected components in the graph generated by the input data

### Basic idea of spectral clustering

In these explanations, we interpret the datapoints as vertices of a weighted graph $G$, with edge weights coming from the affinity matrix $A$.

#### Step 1
Assume for the time being that the graph $G$ is disconnected and consists of two connected components, $G = G_1 \cup G_2$. Let $n$, $n_1$, and $n_2$ be the cardinalities (numbers of vertices) of $G$, $G_1$, and $G_2$ respectively. Now order the vertices so that those in $G_1$ are listed first. Then the corresponding Laplacian $L$ is block diagonal, $L=L_{11}\oplus L_{22}$, with $L_{jj}$ being the Laplacian of the subgraph $G_j$. Therefore, the nullspace of $L$ is two-dimensional, and the basis can be chosen of the form $\mathbf{u}_1 = (v_1, 0)$ and $\mathbf{u}_2 = (0, w_2)$. Form an $n\times 2$ matrix $U$ composed of the columns $\mathbf{u}_1$ and $\mathbf{u}_2$; then the first $n_1$ rows of $U$ are of the form $(v_{1j}, 0)$, while the last $n_2$ rows have the form $(0,w_{2j})$. This helps to cluster the vertices of $G$ in a very simple way using the rows of $U$.

#### Step 2.
Assume now that $G$ is connected, but the components $G_1$ and $G_2$ are only weakly-connected, i.e., the affinity of any vertex of $G_1$ and any vertex of $G_2$ is smaller than within $G_1$ or $G_2$. Then the off-diagonal blocks $L_{12}$ and $L_{21}$ of the Laplacian $L$ are small, so that $L$ can be considered as a small perturbation of the diagonal part $L_0:=L_{11}\oplus L_{22}$, and both the eigenvalues and eigenvectors of $L$ are close to those of $L_0$. Therefore, the lowest two eigenvalues of $G$ have eigenvectors of the form $\mathbf{u}_1 = (v_1,w_1)$ and $\mathbf{u}_2 = (v_2,w_2)$ with small components $w_1$ and $v_2$. As a result, in the first $n_1$ rows $(v_{1j}, v_{2j})$ of the matrix $U$ the second components $v_{2j}$ are much smaller than the first components $v_{1j}$, while in the last $n_2$ rows $(w_{1j},w_{2j})$ the second components dominate. As before, we see that the rows of $U$ can be used to cluster components $G_1$ and $G_2$ in a very efficient way. 

#### Step 3.
In general, the order of the vertices of $G$ is arbitrary; however, if we find the eigenvectors corresponding to two lowest eigenvalues of the Laplacian $L$, form the matrix $U$, normalize its rows, and then perform $2$-means clustering of these rows, then one cluster will consist of rows with dominating first entry, and the other of rows with dominating second entries. These clusters suggest clustering of the vertices of $G$ into parts $G_1$ and $G_2$ with small affinity between $G_1$ and $G_2$. 

#### Step 4.
Clearly, the same arguments give justification of the spectral clustering algorithm when we try to split $G$ into $k$ components. Also, they work for the symmetrized Laplacian $L_{sym}$.

**P.S.** In the case you need further explanations, read ```Section 3``` from [A Tutorial on Spectral Clustering](https://arxiv.org/pdf/0711.0189.pdf) (gives similar reasoning with more detailed and concrete derivation)

### 1.3 Calculating Normalized Laplacian (0.25 pts)

Write a function for calculating the normalized Laplacian from the affinity matrix

In [ ]:
def normalized_laplacian(affinity_matrix: np.array) -> np.array:
    assert len(affinity_matrix.shape) == 2, 'Expected affinity matrix to be 2d array'
    assert affinity_matrix.shape[0] == affinity_matrix.shape[1], 'Expected affinity matrix to be square'
    assert affinity_matrix.shape[0] > 0, 'Too few entries in affinity matrix'
    assert np.sum(np.abs(affinity_matrix - np.transpose(affinity_matrix)) > 1e-6) == 0, 'affinity matrix is not symmetric, sth went wrong in previous steps'
    
    # ========= YOUR CODE STARTS HERE ========= #
    
    # ========== YOUR CODE ENDS HERE ========== #
    
    return normalized_laplacian

### Spectral Clustering! (finally)

At this point, we can finally implement the complete spectral clustering algorithm.

In order to do spectral clustering with $k$ clusters for the dataset $X$, we 

1. compute affinity matrix $A$ of the input data $X$
2. compute normalized Laplacian ($L_{sym}$)
3. find the first $k$ eigenvectors of $L_{sym}$ (i.e. eigenvectors that correspond to $k$ smallest eigenvalues)
4. construct matrix $U \in \mathbb{R}^{N \times k}$ with the columns being eigenvectors from 3. (i.e. $U = [v_1, v_2, ... v_k]$)  
5. normalize the rows $y_i$ of $U$
6. run k-means clustering for points $y_i$

So the labels of kmeans clustering in 6. will give you the label for the original data $X$

**Note:** no need to calculate eigenvalues/eigenvectors by hand

**Note:** no need to do kmeans clustering by hand

**Note:** make sure you instruct your solver to find smallest eigenvalues.

### 1.4 Spectral Clustering Algorithm (0.5 pts)

Implement a function that performs spectral clustering

In [ ]:
def spectral_clustering(X: np.array, affinity_fn: Callable[[np.array], np.array], n_clusters: int, hyperparam: float=1.0) -> np.array:
    # ========= YOUR CODE STARTS HERE ========= #
    
    # ========== YOUR CODE ENDS HERE ========== #
    return labels

In [ ]:
# visualize your clustering results
# ========= YOUR CODE STARTS HERE ========= #

# ========== YOUR CODE ENDS HERE ========== #

### 1.5 Explore hyperparameter space (0.5 pts)

1. Run the spectral clustering algorithm for the dataset introduced above with different parameters:
    * try different values of gamma for **rbf** affinity matrix (e.g. 1, 5, 10, 20)
    * try different values of epsilon for **epsilon-neighbourhood** affinity matrix (you'll need to implement it)
2. Visualize your results


In [ ]:
# gamma analysis
# ========= YOUR CODE STARTS HERE ========= #

# ========== YOUR CODE ENDS HERE ========== #

In [ ]:
def epsilon_affinity(X: np.array, epsilon: float) -> np.array:
    # ========= YOUR CODE STARTS HERE ========= #
    
    # ========== YOUR CODE ENDS HERE ========== #
    return W

In [ ]:
# epsilon analysis
# ========= YOUR CODE STARTS HERE ========= #

# ========== YOUR CODE ENDS HERE ========== #

3. Comment on your results

### 1.6  Conclusions and final comments (0.25 pts)

- Summarize what you learned by completing this lab assignment
- What might be pros and cons of the implemented algorithm
- What is the role of hyperparameters
- Leave any comments on this assignment you may have or your suggestions how it can be improved in the future

## 2. PCA and LDS (2 pts)

**credits to Markiian Novosad for previous version of this assigment!**

In this assignment, the main goal is to discuss **PCA** and **LDA** dimensionality reduction techniques and their performance on classification tasks

A good source to read more on these methods is the book 

*   Kevin P. Murphy. [Probabilistic Machine Learning: An Introduction](https://probml.github.io/pml-book/book1.html), MIT, 2022. Section 9.2.6 (FLDA), 20.1 (PCA and Eigenfaces)

Firstly, we load and prepare our LFW (labeled faces in the wild) dataset, with which we'll work here. This dataset contains images of people faces as well as corresponding labels (unique id of person):

In [ ]:
lfw_people = sklearn.datasets.fetch_lfw_people(min_faces_per_person=100, resize=0.3)
n, h, w = lfw_people.images.shape
labels = lfw_people.target

In [ ]:
class1, class2 =  1, 3 # choose your preferred classes, make sure that there is almost equal number of samples for each class

x = lfw_people.data[(labels == class1)|(labels==class2)]
y = (labels[(labels == class2)|(labels==class1)] == class2)

x_train, x_test, y_train, y_test = sklearn.model_selection.train_test_split(x, y, test_size=0.2, random_state=42)
print("Train x: ", x_train.shape, "Train y: ", y_train.shape, "Test x: ", x_test.shape, "Test x: ", y_test.shape)

Let's visualize our data:

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 6), subplot_kw={'xticks': (), 'yticks': ()})

class1_images = x_train[y_train == 0] # first row of plot for class 0
for i in range(5):
    axes[0, i].imshow(class1_images[i].reshape((h, w)), cmap=plt.cm.gray)
    axes[0, i].set_title(f"Class 0")

class3_images = x_train[y_train == 1] # second row of plot for class 1
for i in range(5):
    axes[1, i].imshow(class3_images[i].reshape((h, w)), cmap=plt.cm.gray)
    axes[1, i].set_title(f"Class 1")

### 2.1 PCA Implementation (0.75 pts)

Implement the code below:

In [ ]:
class MYPCA:
    def __init__(self) -> None:
        pass

    def _get_num_components(self, lambdas, threshold) -> int:
        """
        Calculate minimal number of principal components so that variance explainability >= threshold.
        """
        # ========= YOUR CODE STARTS HERE ========= #

        # ========== YOUR CODE ENDS HERE ========== #
        return num_components

    def center_data(self, data):
        """
        Center our data by subtracting the mean
        """
        # ========= YOUR CODE STARTS HERE ========= #

        # ========== YOUR CODE ENDS HERE ========== #
        return center_data

    def compute_eigen(self, cov_mat):
        """
        Using built-in numpy functionality, compute eigenvectors and eigenvalues of covariance matrix;
        """
        # ========= YOUR CODE STARTS HERE ========= #

        # ========== YOUR CODE ENDS HERE ========== #
        return eigenvalues, eigenvectors

    def fit(self, data, exp_threshold=0.8):
        """
        1. Center the data;
        2. Compute covariance matrix
        3. Compute optimal k (number of components)  
        4. Compute top-k eigenvalues from covariance matrix
        5. Compute components by projecting data with eigenvectors
        """
        # ========= YOUR CODE STARTS HERE ========= #

        # ========== YOUR CODE ENDS HERE ========== #

    def plot_components(self, im_shape):
        """
        For better understanding, visualise the *self.components* variable.
        """
        # ========= YOUR CODE STARTS HERE ========= #

        # ========== YOUR CODE ENDS HERE ========== #

    def plot_explainability(self):
        """
        Plot dependence of variance explainability on number of components.
        """
        # ========= YOUR CODE STARTS HERE ========= #

        # ========== YOUR CODE ENDS HERE ========== #

    def transform(self, data):
        """
        1. Center the data;
        2. Projection onto the components
        """
        # ========= YOUR CODE STARTS HERE ========= #

        # ========== YOUR CODE ENDS HERE ========== #
        return data_trans

### 2.2 PCA analysis (0.25 pts)

In this task, you have to do the following:

* fit your PCA to the traning data
* plot dependency of explainability on number of PCA components used
* plot PCA components
* train SVM (SVC) classifier based on the PCA features of the data, and calculate accuracy on the test set
* discuss your results

In [ ]:
# ========= YOUR CODE STARTS HERE ========= #

# ========== YOUR CODE ENDS HERE ========== #

### Discussion of the results:
* What is the concept of variance explainability and why is it so important?
* What is the optimal number of components to reach good enough classification performance and why? 

### Fischerface classification and LDA

Now, having implemented the **PCA**, let's implement the **Linear Discriminant Analysis** algorithm. By this [link](https://towardsdatascience.com/fishers-linear-discriminant-intuitively-explained-52a1ba79e1bb) you can read more about this algorithm.

In this task, we will see how we can improve our classification accuracy by using **LDA**. 

### 2.3 LDA Implementation (0.5 pts)

In this task, you need to compute the $\mathbf{\it{S_B}}$ and $\mathbf{\it{S_W}}$, where $\mathbf{\it{S_B}}$ is responsible for covariance between classes, and $\mathbf{\it{S_W}}$ is responsible for covariance within classes, and select appropriate eigenvectorsL

In [ ]:
class MYLDA:

    def fit(self, data, data_labels, k_components):
        n_feats = data.shape[-1]
        SW = np.zeros((n_feats, n_feats))
        SB = np.zeros((n_feats, n_feats))
        mu = data.mean(0)
        # ========= YOUR CODE STARTS HERE ========= #

        # ========== YOUR CODE ENDS HERE ========== #
    
    def transform(self, data):
        # ========= YOUR CODE STARTS HERE ========= #

        # ========== YOUR CODE ENDS HERE ========== #
        return proj

Here we will implement the class for the Fischerface algorithm, which basically uses an **LDA** on top of **PCA**, such that we will perform maximum dimensionality reduction for classification purposes:

In [ ]:
class FischerFaces:
    def __init__(self) -> None:
        """
        Initialize pca and lda
        """
        # ========= YOUR CODE STARTS HERE ========= #

        # ========== YOUR CODE ENDS HERE ========== #
        
    def fit(self, X, Y, threshold, lda_components):
        """
        Fit the fischerface algo: 
        1. Fit PCA
        2. Reduce data dimensions with PCA
        3. Fit LDA
        """
        # ========= YOUR CODE STARTS HERE ========= #

        # ========== YOUR CODE ENDS HERE ========== #
        
    def transform(self, X):
        """
        Transform the data by using fitted PCA and LDA
        """
        # ========= YOUR CODE STARTS HERE ========= #

        # ========== YOUR CODE ENDS HERE ========== #
        return lda_trans

### 2.4 LDA & Fisherfaces analysis (0.25 pts)

In this task, you have to do the following:

* fit your FisherFaces to the train data
* train SVM (SVC) classifier based on the FisherFaces features of the data, and calculate accuracy on the test set
* discuss your results

Now, let's try out our new classifier based on Fischerface algorithm. Calculate the accuracy on the test set.

*Hint*: select the number of LDA components equal to the number of classes

In [ ]:
# ========= YOUR CODE STARTS HERE ========= #

# ========== YOUR CODE ENDS HERE ========== #

### Discussion of the results

* Compare results of Fisherfaces and Eigenfaces approaches. Which one performs better and why?

### 2.5 Conclusions and final comments (0.25 pts)

- Summarize what you learned by completing this lab assignment
- What might be pros and cons of the implemented algorithms
- Leave any comments on this assignment you may have or your suggestions how it can be improved in the future

## 3. Point cloud registration. Procrustes problem (2 pts)

**credits to Ostap Viniavskyy for previous version of this assigment!**

The aim of this task is find a rigid transformation that aligns two 3D objects (point clouds) in the best possible way. In the case when point pairs to be matched are given, this is known as Procrustes problem and can be solved via SVD. Otherwise, we will use the Iterative Closest Point (ICP) method that, in most cases, results in good alignment in several iterations

A **point cloud** is a discrete set of data points in space. [Wiki](https://en.wikipedia.org/wiki/Point_cloud) \
In this homework we will work with 3D point clouds representing a 3D shape or object. Each point is associated with its Cartesian coordinate $(X, Y, Z)$ in space and potentially some additional information, like RGB color, even though we would not use any here. Usually, 3D point cloud are obtained from LIDAR scans or from the 3D reconstuction from multiple views of the same scene or object. 

**Point cloud registration** is the problem of finding a spatial transformation (translation, rotation, scaling, etc.) that aligns two point sets. [Wiki](https://en.wikipedia.org/wiki/Point-set_registration) 

<div>
<img src="https://upload.wikimedia.org/wikipedia/commons/f/fe/Cpd_fish_affine.gif" width="300"/>
</div>

In this homework we will work with the case of the **rigid transformation in the 3D space**, meaning that the point clouds in question represent the same 3D rigid body in different positions in space. 

**Rigid-body motion in the 3D space** is represented by the 3x3 rotation matrix $R$ and 3D translation vector $T$. Rotation matrix $R$ belongs to the group of Special-Orthogonal Transformations $SO(3)$ - the group of orthogonal matrices with determinant equalt to $1$. (*Note: Orthogonal matrices whose determinant is equal to $-1$ represent such transformation in space that incorporates both rotation and reflection, the latter of which is not a valid transformation in 3D.*) More on 3D rotation representations and rigid body motion in general can be learnt from this amazing [lecture from the TUM](https://youtu.be/khLM8VV8LuM?list=PLTBdjV_4f-EJn6udZ34tht9EVIW7lbeo4).

Moving to the problem in question, imagine that you have to program an algorithm for a robot to move objects from one point in 3D space to some target position. The robot has the ability to scan the environment around it and obtain the point cloud of the objects. Also, it is provided with some idealized model of the object already in the target position. The main challenge is to find how the scan of real object corresponds to its idealized model and find rigid transformation that provides best alignment.

In the **data** directory you are provided with:
- `bottle_scan/*` -- the scanned water bottle textured mesh from which you can easily extract its point cloud
- `bottle-model.obj` -- the idealized water bottle mesh
- `bottle-model-samples-10k.obj` -- 10k points sampled uniformly on the surface of `bottle-model.obj` mesh in the initial location
- `bottle-model-samples-10k-target.obj` -- same 10k points as in previous file but in the target location
- `bottle-model-samples-8k.obj` -- another 8k points sampled uniformly on the surface of `bottle-model.obj` mesh in the initial location

You can use any 3D viewer to explore the meshes in depth. We recommend using [MeshLab](https://www.meshlab.net/#download) - an open-source tool for 3D objects display and manipulation.

From left to right: photo of the water bottle, scanned water bottle mesh with textures, point cloud of scanned water object, mesh of idealized bottle model, point cloud sampled on the surface of idealized model

![](data/viz.png)

Already, you can see that the idealized model is not exactly the same as the scanned point cloud, and even the number of points is very different. Nevertheless, by carefully step-by-step designing our registration algorithm we will try to align these two point clouds and find good-enough rigid transformation.

Among other problems with point-cloud registration can be:
- partial occlusions;
- outlying points;
- scale differences.

Yet, for simplicity we do not try to solve such cases in this homework. Indeed, classical methods are not very good at aligning partially occluded point clouds, thus Deep Learning methods come to help here.

### The case of known correspondences

We start with the simpler task of aligning two point cloud with known correspondences. They differ only by some unknown rigid-body transformation $R$ and $T$ that you have to find. This problem is known as Orthogonal Procrustes problem as it requires finding an orthogonal 3D rotation matrix $R$. 

For this task we will use point cloud of an idealized model of the water bottle in two different positions in space. The points are sampled on the surface of the model once, and rotated/translated with some true $R$ and $T$ that you have to find. Thus we know exactly the correspondences between 10 thousand points in source and target point clouds.

Here we will be working with the following two files:
- `bottle-model-samples-10k.obj` -- 10k points sampled uniformly on the surface of `bottle-model.obj` mesh in the source location
- `bottle-model-samples-10k-target.obj` -- 10k points sampled uniformly on the surface of `bottle-model.obj` mesh in the target location

Firstly, let's read our source and target meshes:

In [ ]:
def read_obj(fname: str) -> np.ndarray:
    """Read .obj file containing point cloud using trimesh library."""
    with open(fname) as f:
        data = trimesh.exchange.obj.load_obj(f)
    return data['vertices']

src_fname = 'data/bottle-model-samples-10k.obj'
trg_fname = 'data/bottle-model-samples-10k-target.obj'

pts_src = read_obj(src_fname)
pts_trg = read_obj(trg_fname)
print(pts_src.shape, pts_trg.shape)

Given two sets of corresponding points $x_i \leftrightarrow y_i, i \in {1\dots M}, y_i \in \mathbb{R}^3, x_i \in \mathbb{R}^3$ we know that the points locations are related by the following equations:
$$y_i = Rx_i + T$$
Stacking all points into two $3 \times M$ matrices $X$ and $Y$, we can write the equations simultaneously for all points:
$$Y = RX + T_{broadcasted}$$
where $T_{broadcasted}$ is also $3\times M$ matrix of column vector $T$ horizontally stacked $M$ times

In general, we assume that there is some noise present and try to find $R$ and $T$, such that the mean-squared error is minimized:

$$R, T = \arg \min_{R', T'} \sum_i ||y_i - R'x_i + T'||^2$$

First, we can eliminate $T$ from the minimization problem and solve only for $R$. For this reason we normalize point clouds $X$ and $Y$ by subtracting their centroids $\overline{x}$ and $\overline{y}$  from each points in $X$ and $Y$ respectively.
Let's denote the resulting matrices $\hat{X}$ and $\hat{Y}$.

### 3.1 Optimal Translation (0.25 pts)

Find $T'$ and reformulate objective for $R'$ in terms of $\hat{X}$ and $\hat{Y}$:


Once we eliminated $T$, we only look for orthogonal $R$ (actually, special orthogonal R, but it doesn't matter to us now, since we know that true transformation is valid rotation). The problem becomes:

$$R = \arg \min_{R'} \sum_i||\hat{y_i} - R'\hat{x_i}||^2 = \arg \min_{R'} ||R'\hat{X_i} - \hat{Y_i}||^2_F$$

### 3.2 Optimal Rotation (0.25 pts)

Prove that the $R$ that minimizes the above error function is $R = VU^T$, where $\hat{X}\hat{Y}^T = U\Sigma V^T$ is SVD of $3\times 3$ matrix $\hat{X}\hat{Y}^T$:

Then $T$ can be recovered as $T = \overline{y} - R\overline{x}$

### 3.3 Procrustes Implementation (0.25 pts)

Implement solution to the Orthogonal Procrustes problem given two sets of points $X$ and $Y$. You can rely on the numpy SVD solver.

In [ ]:
def orthogonal_procrustes(X: np.ndarray, Y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Implementation of Orthogonal procrustes problem
    Args:
        X: 3D points from the first point cloud; shape: (3, M)
        Y: 3D points from the second point cloud; shape: (3, M)

    Returns:
        R: rotation matrix; shape: (3, 3)
        T: translation vector; shape: (3, )
    """
    # =========== YOUR CODE STARTS HERE ============

    # =========== YOUR CODE ENDS HERE ==============
    return R, T

R_pred, T_pred = orthogonal_procrustes(X=pts_src.T, Y=pts_trg.T)
r_euler = Rotation.from_matrix(R_pred).as_euler('xyz', degrees=True)
print(f'Predicted rotation: X={r_euler[0]:.3}deg, Y={r_euler[1]:.3}deg, Z={r_euler[2]:.3}deg')
print(f'Predicted translation: X={T_pred[0]:.3}m, Y={T_pred[1]:.3}m, Z={T_pred[2]:.3}m')


In [ ]:
# plot predictions X - in blue, Y - in read
def plot_predictions(X, Y, R, T):
    X = X @ R.T + T

    fig = plt.figure(figsize=(17, 17))
    ax = fig.add_subplot(121, projection='3d')
    ax.scatter(X[:, 0], X[:, 1], X[:, 2], s=2., alpha=0.5)
    
    # ax = fig.add_subplot(122, projection='3d')
    ax.scatter(Y[:, 0], Y[:, 1], Y[:, 2], c='r', s=2., alpha=0.5)
    plt.legend(['X', 'Y'])
    
    plt.title('Points clouds X and Y aligned by predicted R_pred, T_pred')
    
    plt.show()
    
plot_predictions(pts_src, pts_trg, R_pred, T_pred)

### The case of unknown correspondences

In this task we will continue to work with model of water bottle. In contrast to the previous problem we will not assume known correspondences, nor equal number of points in source and target point clouds. To simulate this scenario we will sample points on the surface of mesh twice: once 10k points and once 8k points. We will assume some manually specified rigid transformation on the second points cloud and explore how it affects the results of the algorithm.


Here we will be working with the following two files:
- `bottle-model-samples-10k.obj` -- 10k points sampled uniformly on the surface of `bottle-model.obj` mesh in the initial location
- `bottle-model-samples-8k.obj` -- 8k points sampled uniformly on the surface of `bottle-model.obj` mesh in the initial location

Next we will use **Iterative Closest Points (ICP)** algorithm to iteratively find the optimal rigid motion. The algorithm switches between 2 steps:
- finding matching between two sets --> for each $x_i$ in $X$ we find $y_{n_i}$ in the set $Y$ such that $||Rx_i + T - y_{n_i}||^2$ is minimal among all $y_j, j \in 1\dots N$ for current rotation estimate $R$ and translation estimate $T$. 
- solving orthogonal Procrustes problem for the established correspondences to update $R$ and $T$. 

We can define the point-to-point error measure in terms of Mean Squared Error between matched points. 
$MSE(R, T) = \frac{1}{n} \sum_i ||Rx_i + T - y_{n_i}||^2$, where $y_{n_i}$ is the correspondence of $x_i$ as defined above.
Note that each point in $Y$ can be matched with multiple points in $X$.

Repeating those steps, until the error decreases in one iteration less than some threshold $\tau$, will allow us to monotonically decrease the error on each step and converge to a solution. However, the solution is not guaranteed to be the optimal one as the algorithm **may get stuck in the local minima**. 



### 3.4 Correctness of ICP optimization (0.25 pts)

Prove that MSE is guaranteed to monotonically decrease or stay the same on each step of ICP:

### 3.5 ICP Implementation (0.75 pt)

Implement solution to the ICP given two sets of points $X$ and $Y$. You can rely on your previous Orthogonal Procrustes solver. 
Test your solution like the following:
1. Generate ground truth rotations $R_{gt}$ around X-axis with the angles from $0^\circ$ to $355^\circ$ with the step $10^\circ$ and apply it along with provided translation $T_{gt}$ to the `bottle-model-samples-8k.obj` point cloud. You can use scipy rotation representations convertor to obtain rotation matrix from axis-angle representation.
2. Try to estimate the rotation and translation between `bottle-model-samples-10k.obj` and transformed `bottle-model-samples-8k.obj` using your ICP implementation. 
3. Calculate rotation and translation error for each ground truth rotation angle and plot the graphs with angle on the x-axis and error in the y-axis. Code for error calculation will be provided to you. Also plot mse-loss histories and predicted points clouds for the angles of $60^\circ$ and $180^\circ$.
4. Comment on the result. Why the error is nearly 180 degrees after the true rotation exceeds 90 degrees?

- find_optimal_correspondences - 0.25 pts
- icp (optimization loop) - 0.25 pts
- testing on multiple ground truth rotation angles - 0.25 pts
- plots and comments - 0.25 pts


In [ ]:
def find_optimal_correspondences(X: np.ndarray, Y: np.ndarray, 
                                 R: np.ndarray, T: np.ndarray) -> Tuple[np.ndarray, float]:
    """
    Finds closest point from Y for each point in the point cloud X transformed by motion (R, T)
    Also compute mean square error between each transformed point in X and nearest neighbor in Y 
    Args:
        X: 3D points from the first point cloud; shape: (3, M)
        Y: 3D points from the second point cloud; shape: (3, N)
        R: current rotation estimate
        T: current translation estimate

    Returns:
        Y_nn: nearest neighbors from Y for each point in transformed X; shape (3, M)
        J: mean square error
    """
    # =========== YOUR CODE STARTS HERE ============

    # =========== YOUR CODE ENDS HERE ==============
    return Y_nn, J

def icp(X: np.ndarray, Y: np.ndarray, tau: float = 1e-6) -> Tuple[np.ndarray, np.ndarray, List[float]]:
    """
    Implementation of ICP for registration of unmatched point cloud of potentially different sizes
    Args:
        X: 3D points from the first point cloud; shape: (3, M)
        Y: 3D points from the second point cloud; shape: (3, N)
        tau: error decrease threshold used as stopping criterion
    Returns:
        R: rotation matrix; shape: (3, 3)
        T: translation vector; shape: (3, ),
        loss_history: list of MSE value at each iteration
    """
    # init rotation with identity and rotation with the vector between center of point clouds
    R, T = np.eye(3), Y.mean(1) - X.mean(1)
    # compute initial correspondences
    Y_nn, J = find_optimal_correspondences(X, Y, R, T)
    loss_history = [J]
    # =========== YOUR CODE STARTS HERE ============

    # =========== YOUR CODE ENDS HERE ==============
    return R, T, loss_history

In [ ]:
# error metrics
def rigid_motion_error(R_true: np.ndarray, R_pred: np.ndarray,
                       T_true: np.ndarray, T_pred: np.ndarray) -> Tuple[float, float]:
    """
    Calculate error in rigid body motion: rotation and translation separately.
    For rotation compute minimal angle that transforms one rotation into other.
    For translation simply compute distance between two vectors.
    Args:
        R_true: true rotation; shape: (3, 3)
        R_pred: predicted rotation; shape: (3, 3)
        T_true: true translation; shape: (3,)
        T_pred: predicted translation; shape: (3,)
    Returns:
        rot_err: error in rotation in angles
        trans_err: error in translation in meters (or any other units)
    """
    cos = (np.trace(np.dot(R_true.T, R_pred)) - 1) / 2
    cos = np.clip(cos, -1., 1.)  # numercial errors can make it out of bounds
    rot_err = np.rad2deg(np.abs(np.arccos(cos)))
    trans_err = np.linalg.norm(T_true - T_pred)
    return rot_err, trans_err

In [ ]:
# experimets part
src_fname = 'data/bottle-model-samples-10k.obj'
trg_fname = 'data/bottle-model-samples-8k.obj'

pts_src = read_obj(src_fname)
pts_trg = read_obj(trg_fname)
print(pts_src.shape, pts_trg.shape)
# axis of rotation - X
axis = np.array([1., 0., 0.])
# ground  truth translation
T_gt = np.array([0.5, 0.6, 0.7])

# angles
angles = [i * 10 for i in range(36)]
rot_errors = []
trans_errors = []
loss_histories = []

# =========== YOUR CODE STARTS HERE ============

# =========== YOUR CODE ENDS HERE ==============

### 3.6 ICP Testing (0.25 pts)

Test ICP algorithm on real world scan and idealized model as a target. Report the results numerically and visually. Comment on the results.

In [ ]:
src_fname = 'data/bottle_scan/textured_output.obj'
trg_fname = 'data/bottle-model-samples-10k-target.obj'
pts_src = read_obj(src_fname)
pts_trg = read_obj(trg_fname)
print(pts_src.shape, pts_trg.shape)

# =========== YOUR CODE STARTS HERE ===========

# =========== YOUR CODE ENDS HERE ==============

### 3.7 Conclusions and final comments

- Summarize what you learned by completing this lab assignment
- What might be pros and cons of the implemented algorithms
- Leave any comments on this assignment you may have or your suggestions how it can be improved in the future